# Train

### Bibliotecas

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import SMOTE
from feast import FeatureStore

### Carregamento de Dados (Offline) para Treinamento e Testes

In [ ]:
# Primeiro, apontar para o repositório da FeatureStore
off_line_data = FeatureStore(repo_path="feature_repo")

# Em seguida, carregar todos os dados para obter todos os RAs a filtrar dos dados Off-line
training_df = pd.read_parquet("feature_repo/data/df_evasao_escolar.parquet")
print(training_df.columns.tolist())

training_df["DATA_REGISTRO"] = pd.to_datetime(training_df["DATA_REGISTRO"])

# Agora, filtrar a data desejada
data_ref = pd.to_datetime("2022-01-01")

df_filtrado = training_df[training_df["DATA_REGISTRO"].dt.date == data_ref.date()]

df_filtrado.head(-5)

# Criação da entity_df com os RAs únicos e a data de referência
entity_df = df_filtrado[["RA", "DATA_REGISTRO"]].drop_duplicates()
entity_df.head()
#
retrieval_df = off_line_data.get_historical_features(
    entity_df=entity_df,
    features=off_line_data.get_feature_service("aluno_service"),
).to_df()

# Selecionar as features desejadas para o treinamento
features_desejadas = ['EVASAO','DESTAQUE_IEG', 'CG', 'CT', 'DESTAQUE_IPV', 'DESTAQUE_IDA', 'CF',
                       'IDADE', 'FASE_IDEAL', 'FASE', 'ANO_INGRESSO' ]
train_df = retrieval_df[features_desejadas]
train_df.head()

['RA', 'FASE', 'TURMA', 'NOME', 'DATA_NASCIMENTO', 'IDADE', 'GENERO', 'ANO_INGRESSO', 'INSTITUICAO_ENSINO', 'PEDRA', 'INDE', 'CG', 'CF', 'CT', 'IAA', 'IEG', 'IPS', 'REC_PSICOLOGIA', 'IDA', 'MATEMATICA', 'PORTUGUES', 'INGLES', 'INDICADO', 'ATINGIU_PV', 'IPV', 'IAN', 'FASE_IDEAL', 'DEFASAGEM', 'DESTAQUE_IEG', 'DESTAQUE_IDA', 'DESTAQUE_IPV', 'DATA_REGISTRO', 'EVASAO']


,RA,FASE,TURMA,NOME,DATA_NASCIMENTO,IDADE,GENERO,ANO_INGRESSO,INSTITUICAO_ENSINO,PEDRA,...,ATINGIU_PV,IPV,IAN,FASE_IDEAL,DEFASAGEM,DESTAQUE_IEG,DESTAQUE_IDA,DESTAQUE_IPV,DATA_REGISTRO,EVASAO
21,RA-22,6.0,0.0,22,2005.0,17.0,0.0,2019.0,2.0,0.0,...,1.0,8.833,5.0,7.0,-1.0,1.0,1.0,0.0,2022-01-01,1
22,RA-23,6.0,0.0,23,2005.0,17.0,1.0,2021.0,1.0,0.0,...,1.0,8.556,5.0,7.0,-1.0,0.0,1.0,0.0,2022-01-01,0
23,RA-24,6.0,0.0,24,2005.0,17.0,0.0,2021.0,2.0,2.0,...,1.0,9.111,5.0,7.0,-1.0,0.0,0.0,0.0,2022-01-01,1
24,RA-25,6.0,0.0,25,2005.0,17.0,0.0,2021.0,1.0,0.0,...,1.0,9.667,5.0,7.0,-1.0,0.0,1.0,0.0,2022-01-01,0
25,RA-26,6.0,0.0,26,2005.0,17.0,1.0,2022.0,1.0,2.0,...,1.0,10.000,5.0,7.0,-1.0,0.0,0.0,0.0,2022-01-01,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
850,RA-851,0.0,22.0,851,2014.0,8.0,1.0,2022.0,1.0,0.0,...,0.0,6.583,10.0,0.0,0.0,0.0,0.0,1.0,2022-01-01,0
851,RA-852,0.0,23.0,852,2015.0,7.0,0.0,2022.0,1.0,1.0,...,0.0,5.500,10.0,0.0,0.0,1.0,1.0,1.0,2022-01-01,1
852,RA-853,0.0,23.0,853,2014.0,8.0,0.0,2022.0,1.0,2.0,...,1.0,8.500,10.0,0.0,0.0,0.0,0.0,0.0,2022-01-01,0
853,RA-854,0.0,23.0,854,2014.0,8.0,0.0,2022.0,1.0,2.0,...,0.0,7.167,10.0,0.0,0.0,0.0,0.0,1.0,2022-01-01,0


In [25]:
# 2. Separar 10% para validação balanceada (Dados REAIS)
val_size = int(len(train_df) * 0.10)
n_per_class = val_size // 2

df_val_evasao = train_df[train_df['EVASAO'] == 1].sample(n=n_per_class, random_state=42)
df_val_nao_evasao = train_df[train_df['EVASAO'] == 0].sample(n=n_per_class, random_state=42)

df_val = pd.concat([df_val_evasao, df_val_nao_evasao]).sample(frac=1, random_state=42)
df_treino_original = train_df.drop(df_val.index)

X_restante = df_treino_original.drop('EVASAO', axis=1)
y_restante = df_treino_original['EVASAO']

# 3. SMOTE para 10.000 registros balanceados
smote = SMOTE(sampling_strategy={0: 5000, 1: 5000}, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_restante, y_restante)

# 4. Divisão Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# 5. Normalização (Ajustada no treino e aplicada ao resto)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(df_val.drop('EVASAO', axis=1))
y_val = df_val['EVASAO']

# 6. Configuração do GridSearchCV para múltiplos modelos

# Dicionário de modelos e parâmetros
model_params = {
    'knn': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 11],
            'weights': ['uniform', 'distance']
        }
    },
    'logistic_regression': {
        'model': LogisticRegression(max_iter=1000, solver='liblinear'),
        'params': {
            'C': [0.1, 1, 10],
            'penalty': ['l1', 'l2']
        }
    },
    'random_forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_leaf': [1, 4]
        }
    }
}

scores = []

# Loop para rodar os grids
for model_name, mp in model_params.items():
    print(f"Iniciando GridSearchCV para {model_name}...")
    clf = GridSearchCV(mp['model'], mp['params'], cv=5, scoring='f1', n_jobs=-1)
    clf.fit(X_train_scaled, y_train)

    # Predição na validação (Dados REAIS)
    y_pred = clf.predict(X_val_scaled)
    f1 = f1_score(y_val, y_pred)

    scores.append({
        'model': model_name,
        'best_score_treino': clf.best_score_,
        'best_params': clf.best_params_,
        'f1_val_real': f1
    })

# 7. Exibição dos Resultados
print("\n" + "="*50)
print("RESULTADOS FINAIS NA VALIDAÇÃO REAL")
print("="*50)
df_results = pd.DataFrame(scores)
print(df_results[['model', 'f1_val_real', 'best_params']])

# Opcional: Mostrar importância das variáveis para o melhor Random Forest
best_rf = GridSearchCV(model_params['random_forest']['model'],
                       model_params['random_forest']['params'], cv=5, scoring='f1').fit(X_train_scaled, y_train)

importances = pd.Series(best_rf.best_estimator_.feature_importances_, index=X_restante.columns)
print("\nTop 5 Variáveis mais importantes (Random Forest):")
print(importances.sort_values(ascending=False).head(5))

Iniciando GridSearchCV para knn...
Iniciando GridSearchCV para logistic_regression...
Iniciando GridSearchCV para random_forest...


c:\WORK\FIAP-PosMLE\05MLOPS\HandsOn\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\WORK\FIAP-PosMLE\05MLOPS\HandsOn\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



RESULTADOS FINAIS NA VALIDAÇÃO REAL
                 model  f1_val_real  \
0                  knn     0.521739   
1  logistic_regression     0.733333   
2        random_forest     0.492754   

                                         best_params  
0          {'n_neighbors': 3, 'weights': 'distance'}  
1                        {'C': 0.1, 'penalty': 'l1'}  
2  {'max_depth': None, 'min_samples_leaf': 1, 'n_...  

Top 5 Variáveis mais importantes (Random Forest):
CG              0.218073
CF              0.137501
DESTAQUE_IEG    0.100228
CT              0.100146
ANO_INGRESSO    0.090377
dtype: float64
